In [ ]:
import boto3, botocore
from botocore.exceptions import ClientError
import os, time, json, io, zipfile, base64
from datetime import datetime
from dotenv import load_dotenv

import yaml
from misc import load_from_yaml, save_to_yaml
# import iam, s3, lf, rds, aws_networking, ec2

load_dotenv("../env")
boto3.setup_default_session(profile_name="AMShah")

In [ ]:
# from misc import convert_datetime
def convert_datetime(obj):
    if isinstance(obj, datetime):
        return obj.isoformat()
    raise TypeError("Type %s not serializable" % type(obj))


In [ ]:
! echo $AWS_DEFAULT_SUBNET_IDS

In [ ]:
from dotenv import load_dotenv
load_dotenv("../env")
AWS_ALL_IN_ONE_SG = os.environ["AWS_ALL_IN_ONE_SG"]
ACCOUNT_ID        = os.environ['AWS_ACCOUNT_ID_ROOT']
REGION            = os.environ['AWS_DEFAULT_REGION']
VPC_ID            = os.environ['AWS_DEFAULT_VPC']
SECURITY_GROUP_ID = os.environ['AWS_DEFAULT_SG_ID']
SUBNET_IDS        = SUBNET_IDS = os.environ["AWS_DEFAULT_SUBNET_IDS"].split(":")
SUBNET_ID         = SUBNET_IDS[0]
AWS_DEFAULT_KEY_PAIR_NAME = os.environ['AWS_DEFAULT_KEY_PAIR_NAME']
AWS_DEFAULT_INSTANCE_TYPE = os.environ['AWS_DEFAULT_INSTANCE_TYPE']
AWS_DEFAULT_IMAGE_ID = os.environ['AWS_DEFAULT_IMAGE_ID']
AMAZON_LINUX_AMI_ID = os.environ["AMAZON_LINUX_AMI_ID"]

In [ ]:
sts_client           = boto3.client('sts')
rds_client           = boto3.client('rds')
s3_client            = boto3.client('s3')
glue_client          = boto3.client('glue')
lakeformation_client = boto3.client('lakeformation')
stepfunctions_client = boto3.client('stepfunctions')
apigateway_client    = boto3.client('apigateway')
lfn_client           = boto3.client('lambda')
events_client        = boto3.client('events')
sqs_client           = boto3.client('sqs')
iam_client           = boto3.client('iam')


In [ ]:
ec2_client           = boto3.client('ec2', region_name=REGION)
ec2_resource         = boto3.resource('ec2', region_name=REGION)
elbv2_client = boto3.client("elbv2", region_name=REGION)
autoscaling_client = boto3.client("autoscaling", region_name=REGION)
route53_client = boto3.client("route53", region_name=REGION)
acm_client = boto3.client("acm", region_name=REGION)  # Change region as needed

# CloudFormation is a global service, but you can specify a region for the client
cf_client = boto3.client("cloudformation", region_name=REGION)  

# # Example: Get a specific VPC
# vpc = ec2_resource.Vpc('vpc_id')

# # Example: Get a specific EBS volume
# volume = ec2_resource.Volume('volume_id')

[boto3 doc: ELB](https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/elb.html) | 
[boto3 doc: ELBv2](https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/elbv2.html) | 
[boto3 doc: CloudFormation](https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/cloudformation.html) | 
[boto3 doc: autoscaling](https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/autoscaling.html) | 

- [Introduction to AWS Elastic Load Balancers (with hands-on demo)](https://www.youtube.com/watch?v=zCrSCpX4Qec&list=PLReN7WqKlB2ISc0OHamOMSvCUxUValWZU&index=2)
- [Master AWS Auto Scaling with Hands-On Lab | Live Demo for Real-World Skills](https://www.youtube.com/watch?v=3CdR30jjsAA&list=PLReN7WqKlB2ISc0OHamOMSvCUxUValWZU&index=3)
- [Authenticating clients with `mTLS` on Application Load Balancer (ALB) | AWS Events](https://www.youtube.com/watch?v=v8HXlRAHJUE)
- [How can I use an SSL certificate on both my EC2 instance and Elastic Load Balancing?](https://www.youtube.com/watch?v=6Nz0RFfBqVE)

###   [Auto Scaling and Load Balancing on AWS](https://www.youtube.com/watch?v=0mwgbiJae5Q&list=PLO95rE9ahzRs0QMA8qtIAstWFo4X4gHtH&index=5)

#### Option-01: Using Boto3 Library -> Testing Failed!

##### AWS VPC

In [ ]:
vpc_cidr_block = '172.0.0.0/16'
vpc_name = 'asg-alb-vpc'
vpc_id = ec2_client.create_vpc(
    CidrBlock=vpc_cidr_block,
    TagSpecifications=[
        {
            'ResourceType': 'vpc',
            'Tags': [{'Key': 'Name','Value': f"{vpc_name}"},]
        },
    ],
)['Vpc']['VpcId']

print(vpc_id)

In [ ]:
# Check DNS resolution
dns_support = ec2_client.describe_vpc_attribute(VpcId=vpc_id, Attribute="enableDnsSupport")

In [ ]:
# Check DNS hostnames
dns_hostnames = ec2_client.describe_vpc_attribute(VpcId=vpc_id, Attribute="enableDnsHostnames")

In [ ]:
print("DNS Resolution Enabled:", dns_support["EnableDnsSupport"]["Value"])
print("DNS Hostnames Enabled:", dns_hostnames["EnableDnsHostnames"]["Value"])

In [ ]:
# Step 2: Enable DNS resolution
ec2_client.modify_vpc_attribute(VpcId=vpc_id, EnableDnsSupport={"Value": True})

# Step 3: Enable DNS hostnames
ec2_client.modify_vpc_attribute(VpcId=vpc_id, EnableDnsHostnames={"Value": True})

print(f"Created VPC {vpc_id} with DNS support and DNS hostnames enabled.")


In [ ]:
print("DNS Resolution Enabled:", dns_support["EnableDnsSupport"]["Value"])
print("DNS Hostnames Enabled:", dns_hostnames["EnableDnsHostnames"]["Value"])

##### Subnets

In [ ]:
subnet_configs = [
    {'cidr_block': '172.0.1.0/24', 'az': 'us-east-1a', 'tag': 'asg-alb-public-subnet1-us-east-1a'},
    {'cidr_block': '172.0.2.0/24', 'az': 'us-east-1b', 'tag': 'asg-alb-public-subnet2-us-east-1b'},
    {'cidr_block': '172.0.3.0/24', 'az': 'us-east-1a', 'tag': 'asg-alb-private-subnet1-us-east-1a'},
    {'cidr_block': '172.0.4.0/24', 'az': 'us-east-1b', 'tag': 'asg-alb-private-subnet2-us-east-1b'},
]

In [ ]:
public_subnet1 = ec2_resource.create_subnet(
    CidrBlock=subnet_configs[0]['cidr_block'],
    VpcId=vpc_id,
    AvailabilityZone=subnet_configs[0]['az']
)
ec2_client.create_tags(Resources=[public_subnet1.id],Tags=[{'Key': 'Name', 'Value': subnet_configs[0]['tag']}])

public_subnet2 = ec2_resource.create_subnet(
    CidrBlock=subnet_configs[1]['cidr_block'],
    VpcId=vpc_id,
    AvailabilityZone=subnet_configs[1]['az']
)
ec2_client.create_tags(Resources=[public_subnet2.id],Tags=[{'Key': 'Name', 'Value': subnet_configs[1]['tag']}])

In [ ]:
# Enable auto-assign public IPv4
ec2_client.modify_subnet_attribute(SubnetId=public_subnet1.id, MapPublicIpOnLaunch={"Value": True})
ec2_client.modify_subnet_attribute(SubnetId=public_subnet2.id, MapPublicIpOnLaunch={"Value": True})

 -  **How to validate `enable auto-assign public IPv4 address`**:
    -   `public_subnet1` --> `Actions` --> `Edit Subnet Settings` --> `enable auto-assign public IPv4 address` --> `Save`
    -   `public_subnet2` --> `Actions` --> `Edit Subnet Settings` --> `enable auto-assign public IPv4 address` --> `Save`

In [ ]:
private_subnet1 = ec2_resource.create_subnet(
    CidrBlock=subnet_configs[2]["cidr_block"],
    VpcId=vpc_id,
    AvailabilityZone=subnet_configs[2]["az"],
)
ec2_client.create_tags(Resources=[private_subnet1.id],Tags=[{'Key': 'Name', 'Value': subnet_configs[2]['tag']}])

private_subnet2 = ec2_resource.create_subnet(
    CidrBlock=subnet_configs[3]['cidr_block'],
    VpcId=vpc_id,
    AvailabilityZone=subnet_configs[3]['az']
)
ec2_client.create_tags(Resources=[private_subnet2.id],Tags=[{'Key': 'Name', 'Value': subnet_configs[3]['tag']}])

##### Route Tables & Internet Gateway

In [ ]:
# Create Internet Gateway and attach that with VPC
igw = ec2_resource.create_internet_gateway()

In [ ]:
ec2_client.attach_internet_gateway(InternetGatewayId=igw.id,VpcId=vpc_id)

In [ ]:
rtb_public = ec2_resource.create_route_table(VpcId=vpc_id)
ec2_client.create_tags(
    Resources=[rtb_public.id],
    Tags=[{"Key": "Name", "Value": "asg-alb-rtb-public"},],
)

In [ ]:
rtb_public.associate_with_subnet(SubnetId=public_subnet1.id)
rtb_public.associate_with_subnet(SubnetId=public_subnet2.id)

route_params = {"DestinationCidrBlock": "0.0.0.0/0", "GatewayId": igw.id}
rtb_public.create_route(**route_params)

In [ ]:
rtb_private1 = ec2_resource.create_route_table(VpcId=vpc_id)
ec2_client.create_tags(
    Resources=[rtb_private1.id],
    Tags=[
        {"Key": "Name", "Value": "asg-alb-rtb-private1-us-east-1a"},
    ],
)
rtb_private2 = ec2_resource.create_route_table(VpcId=vpc_id)
ec2_client.create_tags(
    Resources=[rtb_private2.id],
    Tags=[
        {"Key": "Name", "Value": "asg-alb-rtb-private2-us-east-1b"},
    ],
)

rtb_private1.associate_with_subnet(SubnetId=private_subnet1.id)
rtb_private2.associate_with_subnet(SubnetId=private_subnet2.id)

##### Security Group

In [ ]:
ALB1_SG_RULES = [
    {
        "IpProtocol": "tcp",
        "FromPort": 80,
        "ToPort": 80,
        "IpRanges": [{"CidrIp": "0.0.0.0/0", "Description": "HTTP_Port"}],
    },
    {
        "IpProtocol": "tcp",
        "FromPort": 443,
        "ToPort": 443,
        "IpRanges": [
            {"CidrIp": "0.0.0.0/0", "Description": "HTTPS_Port"}
        ],  # Open to all IPs
    },
]

# Outbound rules that created automatically by AWS for all security groups.
ALL_IN_ONE_OUTBOUND_RULES = [
    {
        "IpProtocol": "-1",  # '-1' means all protocols
        "FromPort": -1,  # No specific port range (all ports allowed)
        "ToPort": -1,
        "IpRanges": [{"CidrIp": "0.0.0.0/0"}],  # Open to all IPs
    }
]

In [ ]:
ALB1_SG_ID = ec2_client.create_security_group(
    "ALB1_SG",
    vpc_id,
    inbound_rules=ALB1_SG_RULES,
    outbound_rules="",
    tags=[{"Key": "Name", "Value": "ALB1_SG"}],
    description="ALB1_SG",
)["GroupId"]

In [ ]:
# Enable direct access to the Web application (EC2 instance).
WEB1_SG_RULES = [
    {
        "IpProtocol": "tcp",
        "FromPort": 22,  # SSH: For testing only
        "ToPort": 22,
        "IpRanges": [{"CidrIp": "0.0.0.0/0", "Description": "SSH_Port"}],
    },
    {
        "IpProtocol": "tcp",
        "FromPort": 80,  # HTTP: For testing only
        "ToPort": 80,
        "IpRanges": [{"CidrIp": "0.0.0.0/0", "Description": "HTTP_Port"}],
    },
]

WEB1_SG_ID = ec2_client.create_security_group(
    "WEB1_SG",
    vpc_id,
    inbound_rules=WEB1_SG_RULES,
    outbound_rules="",
    tags=[{"Key": "Name", "Value": "WEB1_SG"}],
    description="WEB1_SG",
)["GroupId"]

##### Target Groups

In [ ]:
# Define the target group parameters
TARGET_GROUP_NAME = "TG1"

target_group = elbv2_client.create_target_group(
    Name=TARGET_GROUP_NAME,
    Protocol="HTTP",  # Options: HTTP, HTTPS, TCP, TLS, UDP, TCP_UDP, GENEVE
    Port=80,  # Port on which the target is listening
    VpcId=vpc_id,
    HealthCheckProtocol="HTTP",  # Options: HTTP, HTTPS, TCP, TLS
    HealthCheckPort="traffic-port",  # Use "traffic-port" to match the listener port
    HealthCheckEnabled=True,  # Enable health checks
    HealthCheckPath="/",  # Path for health check requests
    HealthCheckIntervalSeconds=30,  # Interval in seconds between health checks
    HealthCheckTimeoutSeconds=5,  # Timeout before considering the check as failed
    HealthyThresholdCount=3,  # Number of successful checks to consider healthy
    UnhealthyThresholdCount=2,  # Number of failed checks to consider unhealthy
    Matcher={"HttpCode": "200-299"},  # HTTP response code for a successful health check
    TargetType="instance",  # Options: instance, ip, lambda, alb
    Tags=[{"Key": "Environment", "Value": "QA"}],  # Optional tags
)

# Extract the Target Group ARN
TARGET_GROUP_ARN = target_group["TargetGroups"][0]["TargetGroupArn"]
print(f"Target Group created: {TARGET_GROUP_ARN}")

## We will be attaching the Target Group to the Auto Scaling Group in the next step.


##### ACM

In [ ]:
# Define certificate details
domain_name = "harnesstechtx.com"  # Replace with your actual domain
subject_alternative_names = ["www.harnesstechtx.com", "sub.harnesstechtx.com"]  # SANs
validation_method = "DNS"  # Options: "DNS" or "EMAIL"
tags = [{"Key": "Environment", "Value": "QA"}]  # Optional tags
options = {"CertificateTransparencyLoggingPreference": "ENABLED"}  # Enable logging

# Request the ACM Certificate
ssl_certificate_response = acm_client.request_certificate(
    DomainName=domain_name,
    ValidationMethod=validation_method,
    SubjectAlternativeNames=subject_alternative_names,
    # IdempotencyToken="unique-token-123",  # Replace with a unique identifier
    Options=options,
    Tags=tags,
)

# Extract Certificate ARN
SSL_CERTIFICATE_ARN = ssl_certificate_response["CertificateArn"]

- Fetch DNS validation details

In [ ]:
certificate_details = acm_client.describe_certificate(CertificateArn=SSL_CERTIFICATE_ARN)
dns_records = []
for domain_validation in certificate_details["Certificate"]["DomainValidationOptions"]:
    if "ResourceRecord" in domain_validation:
        dns_records.append(domain_validation["ResourceRecord"])

In [ ]:
print(dns_records)

In [ ]:
changes = []
for record in dns_records:
    changes.append(
        {
            "Action": "UPSERT",
            "ResourceRecordSet": {
                "Name": record["Name"],
                "Type": record["Type"],
                "TTL": 300,
                "ResourceRecords": [{"Value": record["Value"]}],
            },
        }
    )

print(changes)


- Adding DNS validation records to Route 53

In [ ]:
# Adding DNS validation records to Route 53
route53_client.change_resource_record_sets(
    HostedZoneId="Z04555692B7PI94BFJEBI",
    ChangeBatch={
        "Comment": "Adding DNS validation records for ACM",
        "Changes": changes,
    },
)

##### Load Balancer

In [ ]:
# ec2_client.modify_vpc_attribute(VpcId=vpc_id, EnableDnsSupport={"Value": True})

# ALB_SSL_ROLE_NAME = "ALB-SSL-Role"

# # Step 4: Create an IAM Role and Policy for SSL (Optional, needed for HTTPS termination)
# ALB_SSL_ROLE_ARN = iam_client.create_role(
#     RoleName=ALB_SSL_ROLE_NAME,
#     AssumeRolePolicyDocument="""{
#         "Version": "2012-10-17",
#         "Statement": [
#             {
#                 "Effect": "Allow",
#                 "Principal": {
#                     "Service": "elasticloadbalancing.amazonaws.com"
#                 },
#                 "Action": "sts:AssumeRole"
#             }
#         ]
#     }""",
# )["Role"]["Arn"]

# iam_client.attach_role_policy(
#     RoleName=ALB_SSL_ROLE_NAME,
#     PolicyArn="arn:aws:iam::aws:policy/service-role/AmazonEC2ContainerServiceRole",
# )

In [ ]:
# Step 7: Create Application Load Balancer
alb_response = elbv2_client.create_load_balancer(
    Name="ALB1",
    Subnets=[public_subnet1.id, public_subnet2.id],
    SecurityGroups=[ALB1_SG_ID],
    Scheme="internet-facing",  # Options: internet-facing, internal
    Type="application",  # Options: application, network, gateway
    IpAddressType="ipv4",  # Options: ipv4, dualstack
)

In [ ]:
print(alb_response)

In [ ]:
ALB_ARN = alb_response["LoadBalancers"][0]["LoadBalancerArn"]
ALB_DNS = alb_response["LoadBalancers"][0]["DNSName"]


# Step 8: Create HTTP Listener (Specify the Target Group)
http_listener_response = elbv2_client.create_listener(
    LoadBalancerArn=ALB_ARN,
    Protocol="HTTP",  # Options: 'HTTP'|'HTTPS'|'TCP'|'TLS'|'UDP'|'TCP_UDP'|'GENEVE'
    Port=80,
    DefaultActions=[{"Type": "forward", "TargetGroupArn": TARGET_GROUP_ARN}],
    # Certificates=[
    #     {"CertificateArn": "string", "IsDefault": True | False},
    # ],
)
HTTP_LISTENER_ARN = http_listener_response["Listeners"][0]["ListenerArn"]

In [ ]:
print(ALB_ARN, ALB_DNS, HTTP_LISTENER_ARN, sep="\n")

In [ ]:
response = elbv2_client.describe_load_balancers(LoadBalancerArns=[ALB_ARN])
print(response)

In [ ]:
print(response["LoadBalancers"][0]["CanonicalHostedZoneId"])
print(yaml.dump(response, default_flow_style=False))

##### Launch Template

-   **IMDS** (Instance Metadata Service) allows EC2 instances to retrieve data about themselves (like instance ID, region, IAM role credentials, etc.).
-   **http://169.254.169.254/latest/meta-data/instance-id** -> This is the EC2 Instance Metadata endpoint — a special internal IP accessible only from within the instance.


-   `$ ssh -i ~/.ssh/AMominNJ.pem ec2-user@75.101.202.156`
-   `$ cd /var/www/html`

In [ ]:
# Define Launch Template parameters
launch_template_name = "MyLT1"
ami_id = "ami-053a45fff0a704a47"  # Amazon Linux 2023 AMI 2023.6.20250211.0 x86_64 HVM kernel-6.1
instance_type = "t2.micro"

# User Data (Base64 Encoded)
user_data_script = """#!/bin/bash

# Update the system and install Apache
yum update -y
yum install -y httpd

# Start and enable Apache to run on boot
systemctl start httpd
systemctl enable httpd

# When working with EC2 instance metadata, the token URL is fixed and standard across all EC2 instances — it always fetches the latest token for accessing EC2 instance metadata
TOKEN=$(curl -X PUT "http://169.254.169.254/latest/api/token" -H "X-aws-ec2-metadata-token-ttl-seconds: 21600")

# Fetch the instance ID
INSTANCEID=$(curl -s http://169.254.169.254/latest/meta-data/instance-id -H "X-aws-ec2-metadata-token: $TOKEN")

# Create or overwrite `index.html` with the instance ID information
echo "<center><h1>This instance has the ID: $INSTANCEID </h1></center>" > /var/www/html/index.html

# Ensure httpd can read the `index.html` file and its directory
chown apache:apache /var/www/html/index.html
chmod 755 /var/www/html
chmod 644 /var/www/html/index.html

# Restart Apache to apply changes
systemctl restart httpd
"""
user_data_encoded = base64.b64encode(user_data_script.encode("utf-8")).decode("utf-8")

# Define Block Device Mappings
block_device_mappings = [
    {
        "DeviceName": "/dev/xvda",
        "Ebs": {
            "VolumeSize": 20,  # 20 GB Root Volume
            "VolumeType": "gp3",
            "DeleteOnTermination": True,
            "Encrypted": True,
        },
    },
    {
        "DeviceName": "/dev/xvdb",
        "Ebs": {
            "VolumeSize": 50,  # Additional 50 GB Volume
            "VolumeType": "gp3",
            "DeleteOnTermination": False,
            "Encrypted": False,
        },
    },
]

launch_template_data = {
    "ImageId": ami_id,
    "InstanceType": instance_type,
    "SecurityGroupIds": [WEB1_SG_ID],
    "UserData": user_data_encoded,
    "BlockDeviceMappings": block_device_mappings,
    # "NetworkInterfaces": [
    #     {
    #         "SubnetId": SUBNET_ID,
    #         "DeviceIndex": 0,
    #         "AssociatePublicIpAddress": True,
    #         "Groups": [WEB1_SG_ID],
    #     }
    # ],
    "TagSpecifications": [
        {
            "ResourceType": "instance",
            "Tags": [{"Key": "Environment", "Value": "QA"}],
        }
    ],
    "Monitoring": {"Enabled": True},  # Enable detailed monitoring
    "DisableApiTermination": False,  # Allow instance termination
    "InstanceInitiatedShutdownBehavior": "stop",  # 'terminate' or 'stop'
    "KeyName": "AMShah",
}

# Create Launch Template
response = ec2_client.create_launch_template(
    LaunchTemplateName=launch_template_name,
    VersionDescription="v1",
    LaunchTemplateData=launch_template_data,
)

# Extract Launch Template ID & ARN
LAUNCH_TEMPLATE_ID = response["LaunchTemplate"]["LaunchTemplateId"]
print(LAUNCH_TEMPLATE_ID)

##### Autoscaling Group

In [ ]:
ASG_NAME = "MyASG"

# Step 2: Create Auto Scaling Group
autoscaling_client.create_auto_scaling_group(
    AutoScalingGroupName=ASG_NAME,
    LaunchTemplate={"LaunchTemplateId": LAUNCH_TEMPLATE_ID, "Version": "$Latest"},
    MinSize=2, # Minimum number of instances
    MaxSize=4, # Maximum number of instances
    DesiredCapacity=2,
    VPCZoneIdentifier=",".join([public_subnet1.id, public_subnet2.id]), #
    HealthCheckType="EC2", # Options: EC2, ELB
    HealthCheckGracePeriod=300, # Wait time before checking health status
    Tags=[{"Key": "Environment", "Value": "QA", "PropagateAtLaunch": True}],
    NewInstancesProtectedFromScaleIn=True, # Protect new instances from scale in
    DefaultCooldown=300, # Default cooldown period in seconds
    AvailabilityZones=["us-east-1a", "us-east-1b"],
    TerminationPolicies=["Default"], # Options: Default, OldestInstance, NewestInstance, OldestLaunchConfiguration, ClosestToNextInstanceHour, AllocationStrategy
    TargetGroupARNs=[TARGET_GROUP_ARN],  # Add ALB target group ARNs if applicable
    InstanceMaintenancePolicy={"MinHealthyPercentage": 50, "MaxHealthyPercentage": 100}, # Instance maintenance policy
)

In [ ]:
# Extract required parts
alb_name, alb_id = ALB_ARN.split("/")[-2:]
tg_name, tg_id = TARGET_GROUP_ARN.split("/")[-2:]

# Construct the correct Resource Label
resource_label = f"app/{alb_name}/{alb_id}/targetgroup/{tg_name}/{tg_id}"
print("Resource Label:", resource_label)


# Step 3: Attach Scaling Policies
scaling_policy_response = autoscaling_client.put_scaling_policy(
    AutoScalingGroupName=ASG_NAME,
    PolicyName="TargetTrackingPolicy",
    PolicyType="TargetTrackingScaling",
    TargetTrackingConfiguration={
        "PredefinedMetricSpecification": {
            "PredefinedMetricType": "ALBRequestCountPerTarget",  # ASGAverageCPUUtilization, ASGAverageNetworkIn, ASGAverageNetworkOut, ALBRequestCountPerTarget
            "ResourceLabel": resource_label,  # Correct format

        },
        "TargetValue": 50.0,
    },
)
print("Scaling Policy Created:", scaling_policy_response["PolicyARN"])

In [ ]:
# # Step 4: Attach Notifications (Optional)
# autoscaling_client.put_notification_configuration(
#     AutoScalingGroupName=ASG_NAME,
#     TopicARN="arn:aws:sns:us-east-1:123456789012:MySNSTopic",
#     NotificationTypes=[
#         "autoscaling:EC2_INSTANCE_LAUNCH",
#         "autoscaling:EC2_INSTANCE_TERMINATE",
#     ],
# )
# print("Notification Configurations Set")

In [ ]:
# # Step 5: Verify Auto Scaling Group Status
# autoscaling_client.describe_auto_scaling_groups(AutoScalingGroupNames=[ASG_NAME])
# # print("Auto Scaling Group Details:", asg_status)

In [ ]:
# print("Resource Label:", resource_label)
# autoscaling_client.describe_policies(AutoScalingGroupName=ASG_NAME)

##### Test Accecibility into your Application

- At this point you should be able to access into your application through IP address of EC2 instance
  - `http://ip_address`

- At this point you should also be able to access into your application through DNS name of Load balancer
  - `http://dns_name_load_balancer`

##### [ Securing AWS Load Balancers with SSL/TLS - Hands-on](https://www.youtube.com/watch?v=R-x1qudp1ig&list=PLO95rE9ahzRs0QMA8qtIAstWFo4X4gHtH&index=6&t=619s)

- Enable access to the web application through Load Balancer Only

In [ ]:
ec2_client.authorize_security_group_ingress(
    GroupId=WEB1_SG_ID,
    IpPermissions=[
        {
            "IpProtocol": "tcp",
            "FromPort": 80,
            "ToPort": 80,
            "UserIdGroupPairs": [
                {
                    "GroupId": ALB1_SG_ID,
                    "Description": "Allow all tcp traffic from ALB1_SG_ID security group on port 80",
                }
            ],
        },
    ],
)


- Revoke direct access to web application

In [ ]:
ec2_client.revoke_security_group_ingress(
    GroupId=WEB1_SG_ID,  # Replace with your Security Group ID
    IpPermissions=[
        {
            "IpProtocol": "tcp",
            "FromPort": 22,  # SSH: For testing only
            "ToPort": 22,
            "IpRanges": [{"CidrIp": "0.0.0.0/0", "Description": "SSH_Port"}],
        },
        {
            "IpProtocol": "tcp",
            "FromPort": 80,  # HTTP: For testing only
            "ToPort": 80,
            "IpRanges": [{"CidrIp": "0.0.0.0/0", "Description": "HTTP_Port"}],
        },
        { # Optional: Remove all traffic from the same security group
            "IpProtocol": "-1",  # '-1' means all protocols
            "UserIdGroupPairs": [
                {
                    "GroupId": WEB1_SG_ID,
                    "Description": "Allow all traffic from THE SAME SECURITY GROUP",
                }
            ],
        },
    ],
)

- At this point you should not be able to access into your application through IP address of EC2.

In [ ]:
# Create HTTPS Listener (Requires an SSL Certificate)
https_listener_response = elbv2_client.create_listener(
    LoadBalancerArn=ALB_ARN,
    Protocol="HTTPS",
    Port=443,
    SslPolicy="ELBSecurityPolicy-TLS13-1-2-Res-2021-06",  # Specify the SSL policy
    Certificates=[{"CertificateArn": SSL_CERTIFICATE_ARN}],
    DefaultActions=[{"Type": "forward", "TargetGroupArn": TARGET_GROUP_ARN}],
)

HTTPS_LISTENER_ARN = https_listener_response["Listeners"][0]["ListenerArn"]

In [ ]:
# Step 10: Wait until ALB is active
while True:
    alb_status = elbv2_client.describe_load_balancers(LoadBalancerArns=[ALB_ARN])
    state = alb_status["LoadBalancers"][0]["State"]["Code"]
    if state == "active":
        break
    print(f"Waiting for ALB to become active... Current state: {state}")
    time.sleep(10)

print("ALB is now active and ready to use!")

# Step 11: Output Load Balancer DNS
print(f"ALB DNS Name: {ALB_DNS}")


In [ ]:
zones = route53_client.list_hosted_zones()["HostedZones"]
for zone in zones:
    if zone["Name"] == "harnesstechtx.com.":
        hosted_zone_id = zone["Id"].split("/")[-1]
print(hosted_zone_id)


In [ ]:
CanonicalHostedZoneId = elbv2_client.describe_load_balancers(LoadBalancerArns=[ALB_ARN])["LoadBalancers"][0]["CanonicalHostedZoneId"]
print(CanonicalHostedZoneId)
print(hosted_zone_id)

In [ ]:
# Create Route53 record set for ALB
res = route53_client.change_resource_record_sets(
    HostedZoneId=hosted_zone_id,
    ChangeBatch={
        "Comment": "Add Alias record",
        "Changes": [
            {
                "Action": "UPSERT",
                "ResourceRecordSet": {
                    "Name": "harnesstechtx.com.", # Must end with a dot
                    "Type": "A",
                    "AliasTarget": {
                        "HostedZoneId": CanonicalHostedZoneId,  # ELB or CloudFront Zone ID
                        "DNSName": ALB_DNS,
                        "EvaluateTargetHealth": True,
                    },
                },
            }
        ],
    },
)

print(json.dumps(res, indent=4, default=convert_datetime))

In [ ]:
# List record sets
response = route53_client.list_resource_record_sets(
    HostedZoneId=hosted_zone_id,
    StartRecordName="harnesstechtx.com.",
    StartRecordType="A",
    MaxItems="1",
)

record_set = response["ResourceRecordSets"][0]
print(yaml.dump(record_set, indent=4))


In [ ]:
# Delete the existing A record
res = route53_client.change_resource_record_sets(
    HostedZoneId=hosted_zone_id,
    ChangeBatch={
        "Comment": "Add Alias record",
        "Changes": [
            {
                "Action": "DELETE",
                "ResourceRecordSet": {
                    "Name": "harnesstechtx.com.",  # Must end with a dot
                    "Type": "A",
                    "AliasTarget": {
                        "HostedZoneId": CanonicalHostedZoneId,  # ELB or CloudFront Zone ID
                        "DNSName": ALB_DNS,
                        "EvaluateTargetHealth": True,
                    },
                },
            }
        ],
    },
)
print(json.dumps(res, indent=4, default=convert_datetime))

In [ ]:
# Get all listeners for the specified ALB
response = elbv2_client.describe_listeners(LoadBalancerArn=ALB_ARN)

# Loop through the listeners and find the one using HTTP (port 80)
for listener in response["Listeners"]:
    if listener["Port"] == 80 and listener["Protocol"] == "HTTP":
        http_listener_arn = listener["ListenerArn"]
        print("HTTP Listener ARN:", http_listener_arn)


In [ ]:
# NOT TESTED YET
# Create a redirect rule on the HTTP listener
response = elbv2_client.create_rule(
    ListenerArn=http_listener_arn,
    Conditions=[
        {"Field": "path-pattern", "Values": ["/*"]},
    ],
    Priority=1,  # must be unique and > default rule (usually 1-50000)
    Actions=[
        {
            "Type": "redirect",
            "RedirectConfig": {
                "Protocol": "HTTPS",
                "Port": "443",
                "StatusCode": "HTTP_301",  # or 'HTTP_302'
            },
        }
    ],
)

##### Delete Resources

In [ ]:
elbv2_client.delete_load_balancer(LoadBalancerArn=ALB_ARN)

In [ ]:
elbv2_client.delete_target_group(TargetGroupArn=TARGET_GROUP_ARN)

In [ ]:
ec2_client.delete_launch_template(LaunchTemplateId=LAUNCH_TEMPLATE_ID)

In [ ]:
autoscaling_client.delete_auto_scaling_group(AutoScalingGroupName=ASG_NAME,ForceDelete=True)

In [ ]:
aws_networking.delete_vpc_with_dependencies(vpc_id)

In [ ]:
acm_client.delete_certificate(CertificateArn=SSL_CERTIFICATE_ARN)

#### Option-02: Using CloudFormation Template through Boto3 Library.

<div> CloudFormation Templates: Click Here to view
    <div style="display:none">
    <pre><code>
    # [Securing AWS Load Balancers with SSL/TLS - Hands-on](https://www.youtube.com/watch?v=R-x1qudp1ig)

    AWSTemplateFormatVersion: "2010-09-09"
    Description: Deploy an ASG with instances across two AZs and an ALB in front.
    Parameters:
        VPCId:
            Description: Select the VPC
            Type: AWS::EC2::VPC::Id
        SubnetIdOne:
            Description: Select the first subnet for the ALB and Instance
            Type: AWS::EC2::Subnet::Id
        SubnetIdTwo:
            Description: Select the second subnet for the ALB and Instance
            Type: AWS::EC2::Subnet::Id
        ImageId:
            Description: AMI ID for the EC2 instances
            Type: AWS::EC2::Image::Id
        InstanceType:
            Description: EC2 instance type
            Type: String
            Default: t2.micro
    Resources:
        ALBSecurityGroup:
            Type: AWS::EC2::SecurityGroup
            Properties:
                GroupDescription: Allow HTTP to the load balancer
                VpcId: !Ref VPCId
                SecurityGroupIngress:
                    - IpProtocol: tcp
                    FromPort: 80
                    ToPort: 80
                    CidrIp: 0.0.0.0/0
        InstanceSecurityGroup:
            Type: AWS::EC2::SecurityGroup
            Properties:
                GroupDescription: Allow HTTP from the ALB
                VpcId: !Ref VPCId
                SecurityGroupIngress:
                    - IpProtocol: tcp
                    FromPort: 80
                    ToPort: 80
                    SourceSecurityGroupId: !Ref ALBSecurityGroup
        LaunchTemplate:
            Type: AWS::EC2::LaunchTemplate
            Properties:
                LaunchTemplateName: MyLaunchTemplate
                LaunchTemplateData:
                    ImageId: !Ref ImageId
                    InstanceType: !Ref InstanceType
                    SecurityGroupIds:
                        - !Ref InstanceSecurityGroup
                    UserData: !Base64 |
                        #!/bin/bash
                        yum update -y
                        yum install -y httpd
                        systemctl start httpd
                        systemctl enable httpd
                        # Fetch the token for IMDSv2
                        TOKEN=`curl -X PUT "http://169.254.169.254/latest/api/token" -H "X-aws-ec2-metadata-token-ttl-seconds: 21600" || echo ""`
                        # Use the token to fetch the availability zone
                        if [ -n "$TOKEN" ]; then
                        AZ=`curl -H "X-aws-ec2-metadata-token: $TOKEN" -s http://169.254.169.254/latest/meta-data/placement/availability-zone`
                        else
                        AZ="Unknown"
                        fi
                        echo "<h1>This instance is in Availability Zone: $AZ</h1>" > /var/www/html/index.html
        AutoScalingGroup:
            Type: AWS::AutoScaling::AutoScalingGroup
            Properties:
                AutoScalingGroupName: MyAutoScalingGroup
                MinSize: "1"
                MaxSize: "2"
                DesiredCapacity: "2"
                LaunchTemplate:
                    LaunchTemplateId: !Ref LaunchTemplate
                    Version: !GetAtt LaunchTemplate.LatestVersionNumber
                VPCZoneIdentifier:
                    - !Ref SubnetIdOne
                    - !Ref SubnetIdTwo
                TargetGroupARNs:
                    - !Ref TargetGroup
        LoadBalancer:
            Type: AWS::ElasticLoadBalancingV2::LoadBalancer
            Properties:
                Subnets:
                    - !Ref SubnetIdOne
                    - !Ref SubnetIdTwo
                SecurityGroups:
                    - !Ref ALBSecurityGroup
        TargetGroup:
            Type: AWS::ElasticLoadBalancingV2::TargetGroup
            Properties:
                Port: 80
                Protocol: HTTP
                VpcId: !Ref VPCId
                HealthCheckEnabled: true
                HealthCheckPath: /
                Matcher:
                    HttpCode: "200"
        Listener:
            Type: AWS::ElasticLoadBalancingV2::Listener
            Properties:
                DefaultActions:
                    - Type: forward
                    TargetGroupArn: !Ref TargetGroup
                LoadBalancerArn: !Ref LoadBalancer
                Port: 80
                Protocol: HTTP
    </code></pre>
    </div>
</dev>

In [ ]:
# Define stack name
stack_name = "securing-alb-with-tls-certificates"

# Read CloudFormation template from a file
with open(
    "/Users/am/mydocs/Software_Development/Web_Development/aws/boto_scripts/alb-cf-template.yml",
    "r",
) as alb_cf_template:
    template_body = alb_cf_template.read()

# Define CloudFormation parameters
parameters = [
    {
        "ParameterKey": "VPCId",
        "ParameterValue": os.environ["AWS_DEFAULT_VPC"],
    },
    {
        "ParameterKey": "SubnetIdOne",
        "ParameterValue": os.environ["AWS_DEFAULT_SUBNET_A"],
    },
    {
        "ParameterKey": "SubnetIdTwo",
        "ParameterValue": os.environ["AWS_DEFAULT_SUBNET_C"],
    },
    {
        "ParameterKey": "ImageId",
        "ParameterValue": os.environ["AMAZON_LINUX_AMI_ID"],
    },
    {   
        "ParameterKey": "InstanceType", 
        "ParameterValue": "t2.micro"
    },
]

# Create CloudFormation stack
print(f"Creating CloudFormation stack: {stack_name}")
response = cf_client.create_stack(
    StackName=stack_name,
    TemplateBody=template_body,
    Parameters=parameters,
    # Capabilities=["CAPABILITY_IAM", "CAPABILITY_NAMED_IAM"],  # Required for IAM resources only if your template creates IAM resources
)


### MISC

#### `create_listener(...)` with Client Authentication (mTLS)

In [ ]:
# NOT TESTED YET
# Create an HTTPS listener with mTLS (Mutual TLS) on the Application Load Balancer
response = elbv2_client.create_listener(
    LoadBalancerArn="arn:aws:elasticloadbalancing:region:account-id:loadbalancer/app/my-load-balancer/50dc6c495c0c9188",
    Protocol="HTTPS",
    Port=443,
    Certificates=[
        {
            "CertificateArn": "arn:aws:acm:region:account-id:certificate/your-server-cert-id"
        }
    ],
    SslPolicy="ELBSecurityPolicy-TLS13-1-2-2021-06",  # Choose a secure policy supporting mTLS
    DefaultActions=[
        {
            "Type": "forward",
            "TargetGroupArn": "arn:aws:elasticloadbalancing:region:account-id:targetgroup/my-targets/73e2d6bc24d8a067",
        }
    ],
    AlpnPolicy=["HTTP1Only"],  # Optional: ALPN for protocol negotiation
    ClientCertificateTlsAuth={
        "Mode": "require",  # or 'optional'
        "TrustStoreArn": "arn:aws:acm-pca:region:account-id:truststore/your-truststore-id",
        "ValidationMode": "strict",  # or 'default' – strict ensures strong validation
    },
)

print("HTTPS Listener with mTLS created:", response["Listeners"][0]["ListenerArn"])
